<a href="https://colab.research.google.com/github/AHMADAMJAD0/NLP_with_Olga/blob/main/NLP_task_2_NOV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ============================================================
# SYNTAGMATIC RELATION PRACTICAL
# Target word: "time"
# Output:
# 1. entropy_results.csv
# 2. mi_results.csv
# 3. mi_verbs_results.csv
# ============================================================

import re
import math
import nltk
import pandas as pd

from collections import Counter
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet


# ------------------------------------------------------------
# STEP 1: Download NLTK resources
# ------------------------------------------------------------

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("wordnet")
nltk.download("omw-1.4")


# ------------------------------------------------------------
# STEP 2: Read text file
# ------------------------------------------------------------

file_path = "/content/text_news.txt"

with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
    raw_text = file.read()

print("Raw text loaded.")


# ------------------------------------------------------------
# STEP 3: Clean corpus noise
# ------------------------------------------------------------

# Remove document IDs such as @@3000041
raw_text = re.sub(r"@@\d+", " ", raw_text)

# Remove <p>
raw_text = raw_text.replace("<p>", " ")

# Remove repeated @ symbols
raw_text = re.sub(r"@\s*@", " ", raw_text)

# Remove extra spaces
raw_text = re.sub(r"\s+", " ", raw_text)


# ------------------------------------------------------------
# STEP 4: Sentence tokenization
# ------------------------------------------------------------

sentences = sent_tokenize(raw_text)

print("Number of sentences:", len(sentences))


# ------------------------------------------------------------
# STEP 5: Lemmatizer
# ------------------------------------------------------------

lemmatizer = WordNetLemmatizer()


# Convert POS tag to WordNet POS
def get_wordnet_pos(tag):

    if tag.startswith("J"):
        return wordnet.ADJ

    elif tag.startswith("V"):
        return wordnet.VERB

    elif tag.startswith("N"):
        return wordnet.NOUN

    elif tag.startswith("R"):
        return wordnet.ADV

    else:
        return wordnet.NOUN


# ------------------------------------------------------------
# STEP 6: Normalize every sentence
# lowercase + POS tagging + lemmatization
# ------------------------------------------------------------

normalized_sentences = []

verbs = set()


for sentence in sentences:

    # tokenize
    tokens = word_tokenize(sentence.lower())

    # keep alphabetic words only
    tokens = [
        word
        for word in tokens
        if word.isalpha()
    ]

    # POS tagging
    tagged_words = pos_tag(tokens)

    normalized_words = []

    for word, tag in tagged_words:

        pos = get_wordnet_pos(tag)

        # lemmatization
        lemma = lemmatizer.lemmatize(
            word,
            pos
        )

        normalized_words.append(lemma)

        # Save verbs separately
        if tag.startswith("V"):
            verbs.add(lemma)

    # set() because we only care if word occurs
    # in a sentence, not how many times in that sentence
    normalized_sentences.append(
        set(normalized_words)
    )


print("Normalization completed.")


# ------------------------------------------------------------
# STEP 7: Target word
# ------------------------------------------------------------

TARGET = "time"


# Number of sentences
N = len(normalized_sentences)


# ------------------------------------------------------------
# STEP 8: Count sentence occurrences
# ------------------------------------------------------------

word_sentence_count = Counter()

target_and_word_count = Counter()

target_count = 0


for sentence in normalized_sentences:

    # Count each word once per sentence
    for word in sentence:

        word_sentence_count[word] += 1

    # Check whether target occurs
    if TARGET in sentence:

        target_count += 1

        # Count words occurring with target
        for word in sentence:

            if word != TARGET:
                target_and_word_count[word] += 1


print("Sentences containing 'time':", target_count)


# ------------------------------------------------------------
# STEP 9: Laplace smoothing
# ------------------------------------------------------------

# Add-one smoothing
ALPHA = 1.0


# ------------------------------------------------------------
# STEP 10: Calculate Entropy and Mutual Information
# ------------------------------------------------------------

results = []


for word in word_sentence_count:

    if word == TARGET:
        continue

    # Ignore single-character noise
    if len(word) <= 1:
        continue


    # --------------------------------------------------------
    # 2 x 2 contingency table
    # --------------------------------------------------------

    # time = 1, word = 1
    n11 = target_and_word_count[word]

    # time = 1, word = 0
    n10 = target_count - n11

    # time = 0, word = 1
    n01 = word_sentence_count[word] - n11

    # time = 0, word = 0
    n00 = N - n11 - n10 - n01


    # --------------------------------------------------------
    # Add-one smoothing
    # --------------------------------------------------------

    n11 = n11 + ALPHA
    n10 = n10 + ALPHA
    n01 = n01 + ALPHA
    n00 = n00 + ALPHA


    total = (
        n11 +
        n10 +
        n01 +
        n00
    )


    # --------------------------------------------------------
    # Joint probabilities
    # --------------------------------------------------------

    p11 = n11 / total
    p10 = n10 / total
    p01 = n01 / total
    p00 = n00 / total


    # --------------------------------------------------------
    # Marginal probabilities
    # --------------------------------------------------------

    # P(time = 1)
    p_w1_1 = p11 + p10

    # P(time = 0)
    p_w1_0 = p01 + p00

    # P(word = 1)
    p_w2_1 = p11 + p01

    # P(word = 0)
    p_w2_0 = p10 + p00


    # --------------------------------------------------------
    # CONDITIONAL ENTROPY H(time | word)
    # --------------------------------------------------------

    entropy = 0


    # word = 1
    p_time1_given_word1 = p11 / p_w2_1
    p_time0_given_word1 = p01 / p_w2_1


    entropy_word1 = -(
        p_time1_given_word1 *
        math.log2(p_time1_given_word1)
        +
        p_time0_given_word1 *
        math.log2(p_time0_given_word1)
    )


    # word = 0
    p_time1_given_word0 = p10 / p_w2_0
    p_time0_given_word0 = p00 / p_w2_0


    entropy_word0 = -(
        p_time1_given_word0 *
        math.log2(p_time1_given_word0)
        +
        p_time0_given_word0 *
        math.log2(p_time0_given_word0)
    )


    # Final conditional entropy
    entropy = (
        p_w2_1 * entropy_word1
        +
        p_w2_0 * entropy_word0
    )


    # --------------------------------------------------------
    # MUTUAL INFORMATION
    # --------------------------------------------------------

    mi = 0


    mi += p11 * math.log2(
        p11 /
        (p_w1_1 * p_w2_1)
    )

    mi += p10 * math.log2(
        p10 /
        (p_w1_1 * p_w2_0)
    )

    mi += p01 * math.log2(
        p01 /
        (p_w1_0 * p_w2_1)
    )

    mi += p00 * math.log2(
        p00 /
        (p_w1_0 * p_w2_0)
    )


    # Save result
    results.append({

        "word": word,

        "time_count":
            target_count,

        "word_count":
            word_sentence_count[word],

        "time_and_word_count":
            target_and_word_count[word],

        "conditional_entropy":
            entropy,

        "mutual_information":
            mi
    })


# ------------------------------------------------------------
# STEP 11: Convert to DataFrame
# ------------------------------------------------------------

df = pd.DataFrame(results)


# ============================================================
# FILE 1 — CONDITIONAL ENTROPY
# Lower entropy = stronger predictability
# ============================================================

entropy_df = df.sort_values(
    "conditional_entropy",
    ascending=True
)

entropy_df.to_csv(
    "/content/entropy_results.csv",
    index=False
)


# ============================================================
# FILE 2 — MUTUAL INFORMATION
# Higher MI = stronger association
# ============================================================

mi_df = df.sort_values(
    "mutual_information",
    ascending=False
)

mi_df.to_csv(
    "/content/mi_results.csv",
    index=False
)


# ============================================================
# FILE 3 — MI ONLY FOR VERBS
# ============================================================

mi_verbs_df = mi_df[
    mi_df["word"].isin(verbs)
]

mi_verbs_df.to_csv(
    "/content/mi_verbs_results.csv",
    index=False
)


# ------------------------------------------------------------
# STEP 12: Show results
# ------------------------------------------------------------

print("\n==============================")
print("TOP 15 BY MUTUAL INFORMATION")
print("==============================")

print(
    mi_df[
        [
            "word",
            "word_count",
            "time_and_word_count",
            "mutual_information"
        ]
    ].head(15)
)


print("\n==============================")
print("TOP 15 VERBS BY MI")
print("==============================")

print(
    mi_verbs_df[
        [
            "word",
            "word_count",
            "time_and_word_count",
            "mutual_information"
        ]
    ].head(15)
)


print("\nFiles created successfully:")

print("/content/entropy_results.csv")
print("/content/mi_results.csv")
print("/content/mi_verbs_results.csv")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Raw text loaded.
Number of sentences: 62792
Normalization completed.
Sentences containing 'time': 1997

TOP 15 BY MUTUAL INFORMATION
       word  word_count  time_and_word_count  mutual_information
19       at        6248                  505            0.004638
11      the       34098                 1463            0.003594
112   first        1487                  165            0.002258
1228   same         576                   96            0.002070
170    york         437                   78            0.001796
48      for        9331                  502            0.001716
21       to       20708                  903            0.001533
1688  spend         325                   62            0.001521
280      it        9384                  472            0.001247
56     this        3598                  230            0.001178
498    long         661                   76            0.001082
60       be       25604                 1025            0.001073
247     all        222